# 03 — CU1: ResNet-50 + EfficientNet-B0, y fusión imagen + texto

Fine-tuning completo de ResNet-50 (cu1-resnet50-v1) y EfficientNet-B0
(cu1-efficientnet-b0-v1) desde ImageNet. Meta (OE1): accuracy ≥ 0,80 y
F1-macro ≥ 0,75 en el conjunto de prueba.

Corre `entrenar_cu1.py` y después `evaluar_fusion.py`, que elige en
validación el peso de la imagen al combinarla con el texto (con la misma
fórmula que usa ai-service) — ese valor va en `PESO_FUSION_IMAGEN` de
`ai-service/.env`. Corre antes los notebooks 01 y/o 02: la fusión
necesita un modelo de texto entrenado.

Tiempo estimado con GPU T4: del orden de 45-90 minutos para las dos
arquitecturas. Cada mejora se guarda en Drive (`modelo.pt`): si Colab se
desconecta, no se pierde el mejor checkpoint hasta ese momento.

In [ ]:
# Si ya corriste esta celda en esta sesión de Colab, puedes saltarla.
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_ML = "/content/drive/MyDrive/NovaStore_IA/ml"
    get_ipython().system(f"pip install -q -r {CARPETA_ML}/requirements.txt")
else:
    # Ejecución local (verificación, no Colab): la carpeta ml/ del repo.
    CARPETA_ML = str(Path.cwd())

os.chdir(CARPETA_ML)
sys.path.insert(0, CARPETA_ML)
print("CARPETA_ML =", CARPETA_ML)


In [ ]:
MODO_RAPIDO = False  # True = pocas filas y 1 época: solo para probar que corre (métricas sin valor)

if MODO_RAPIDO:
    # Si todavía no corriste 00_preparar_dataset, genera un dataset sintético
    # diminuto para poder probar la plomería (no sirve para medir nada).
    from _modo_rapido_fallback import generar_si_falta
    generar_si_falta(Path(CARPETA_ML) / "datos")

import torch
print("GPU disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("Sin GPU: en Colab, activa Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU.")


## Entrenar ResNet-50 y EfficientNet-B0

In [ ]:
get_ipython().system("python entrenar_cu1.py" + (" --modo-rapido" if MODO_RAPIDO else " --reanudar"))


## Resultados por arquitectura

In [ ]:
import json
from IPython.display import Image as MostrarImagen, display

carpeta = Path(CARPETA_ML) / "modelos" / "cu1-resnet50-v1"
metricas = json.loads((carpeta / "metricas.json").read_text(encoding="utf-8"))
print({k: v for k, v in metricas.items() if k in ("accuracy", "f1_macro", "f1_weighted", "n")})
print("Por fuente:", json.dumps(metricas.get("por_fuente", {}), indent=2, ensure_ascii=False))
for nombre in ("curvas_entrenamiento.png", "matriz_confusion.png"):
    if (carpeta / nombre).exists():
        display(MostrarImagen(filename=str(carpeta / nombre)))


In [ ]:
import json
from IPython.display import Image as MostrarImagen, display

carpeta = Path(CARPETA_ML) / "modelos" / "cu1-efficientnet-b0-v1"
metricas = json.loads((carpeta / "metricas.json").read_text(encoding="utf-8"))
print({k: v for k, v in metricas.items() if k in ("accuracy", "f1_macro", "f1_weighted", "n")})
print("Por fuente:", json.dumps(metricas.get("por_fuente", {}), indent=2, ensure_ascii=False))
for nombre in ("curvas_entrenamiento.png", "matriz_confusion.png"):
    if (carpeta / nombre).exists():
        display(MostrarImagen(filename=str(carpeta / nombre)))


## Fusión imagen + texto

In [ ]:
get_ipython().system("python evaluar_fusion.py")
